<a href="https://colab.research.google.com/github/Raymondycp/QuantFinance_AlgoTradingStrategy/blob/main/%5BQuantFinance%5DSma_FineTurningParameter_Backtesting_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install backtesting ipywidgets


In [ ]:
import ipywidgets
print(ipywidgets.__version__)

In [ ]:
import datetime
import warnings

import pandas as pd
import requests
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import SMA


pd.set_option("display.max_columns", None)

In [ ]:
import datetime
import requests
import pandas as pd

stock_index = "TSLA"
url = "https://api.finmindtrade.com/api/v4/data"
parameter = {
    "dataset": "USStockPrice",  # Changed to US stock dataset
    "start_date": datetime.datetime(2024, 1, 1).strftime("%Y-%m-%d"),
    "end_date": datetime.datetime(2025, 1, 1).strftime("%Y-%m-%d"),
    "data_id": stock_index,
}

data = requests.get(url, params=parameter)
data = data.json()

df = pd.DataFrame(data["data"])

df.index = pd.to_datetime(df["date"])
df.rename(
    columns={
        "volume": "Volume",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
    },
    inplace=True,
)

# Drop unnecessary columns (adjust based on actual API response)
df.drop(
    columns=["stock_id", "date"],  # Add any other columns you don't need
    inplace=True,
)

print(df.head())

In [ ]:
df.info()

In [ ]:
class SmaCross(Strategy):
    def init(self):
        self.fast_line = self.I(SMA, self.data.Close,7)
        self.slow_line = self.I(SMA, self.data.Close,10)

    def next(self):
        if crossover(self.fast_line, self.slow_line):
            print(
                f"{self.data.index[-1]} Buy: Price: {self.data.Close[-1]}, Slow: {self.slow_line[-5:]}, Fast: {self.fast_line[-5:]}"
            )
            self.buy()
        elif crossover(self.slow_line, self.fast_line):
            print(
                f"{self.data.index[-1]} Sell: Price: {self.data.Close[-1]}, Slow: {self.slow_line[-5:]}, Fast: {self.fast_line[-5:]}"
            )

            self.sell()

In [ ]:
test = Backtest(
    df,
    SmaCross,
    cash=10000,
    commission=0.004,
    exclusive_orders=True,
    trade_on_close=True,
)
result = test.run()

In [18]:
print(result)

Start                     2024-01-02 00:00:00
End                       2024-12-31 00:00:00
Duration                    364 days 00:00:00
Exposure Time [%]                    89.28571
Equity Final [$]                   8457.28928
Equity Peak [$]                   10675.61448
Commissions [$]                     1350.3312
Return [%]                          -15.42711
Buy & Hold Return [%]                83.63876
Return (Ann.) [%]                   -15.42711
Volatility (Ann.) [%]                54.03383
CAGR [%]                            -10.95253
Sharpe Ratio                         -0.28551
Sortino Ratio                        -0.36207
Calmar Ratio                         -0.25205
Alpha [%]                            -11.6873
Beta                                 -0.04471
Max. Drawdown [%]                   -61.20706
Avg. Drawdown [%]                   -14.13937
Max. Drawdown Duration      305 days 00:00:00
Avg. Drawdown Duration       66 days 00:00:00
# Trades                          

In [ ]:
result.tail()

In [ ]:
test.plot()

# Fine turning parameter

In [ ]:
import pandas as pd
from backtesting import Backtest
from backtesting import Strategy
from backtesting.lib import crossover

class SmaCross(Strategy):
    # Define optimizable parameters
    n1 = 10  # Fast SMA default value
    n2 = 20  # Slow SMA default value

    def init(self):
        # Calculate two SMAs
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        # Buy when fast SMA crosses above slow SMA
        if crossover(self.sma1, self.sma2):
            self.buy()
        # Sell when fast SMA crosses below slow SMA
        elif crossover(self.sma2, self.sma1):
            self.sell()

# Simple Moving Average function
def SMA(prices, n):
    return pd.Series(prices).rolling(n).mean()

# Assume df is your data
# df = pd.read_csv('your_data.csv', parse_dates=True, index_col=0)

# Create backtest object
bt = Backtest(df, SmaCross, cash=10000, commission=0.004,
              exclusive_orders=True, trade_on_close=True)

# Set parameter ranges
param_grid = {
    'n1': range(5, 50, 5),    # Fast SMA range: 5 to 45, step 5
    'n2': range(10, 100, 10)  # Slow SMA range: 10 to 90, step 10
}

# Run optimization targeting Sharpe Ratio
optimization_results = bt.optimize(
    **param_grid,
    maximize='Sharpe Ratio',  # Maximize Sharpe Ratio
    constraint=lambda param: param.n1 < param.n2,  # Ensure n1 < n2
    return_heatmap=False       # Don't return heatmap data to save memory
)

# Convert optimization results to DataFrame and sort by Sharpe Ratio descending
results_df = optimization_results.sort_values(by='Sharpe Ratio', ascending=False)

# Print top 5 best results
print("Top 5 parameter combinations with best Sharpe Ratio:")
print(results_df.head(5)[['n1', 'n2', 'Sharpe Ratio']].to_string(index=False))

# Print complete backtest results for best parameter combination
best_result = results_df.iloc[0]
print("\nComplete results for best parameter combination:")
print(f"Fast SMA (n1): {best_result['n1']}")
print(f"Slow SMA (n2): {best_result['n2']}")
print(f"Sharpe Ratio: {best_result['Sharpe Ratio']:.2f}")
print(f"Annual Return: {best_result['Return [%]']:.2f}%")
print(f"Max Drawdown: {best_result['Max. Drawdown [%]']:.2f}%")

# Multithread

In [ ]:
from multiprocessing import cpu_count

cpu_count()

In [19]:
import pandas as pd
from backtesting import Backtest
from backtesting import Strategy
from backtesting.lib import crossover
import multiprocessing as mp
from tqdm import tqdm  # Progress bar display

class SmaCross(Strategy):
    n1 = 2  # Fast SMA default value
    n2 = 3  # Slow SMA default value

    def init(self):
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        if crossover(self.sma1, self.sma2):
            self.buy()
        elif crossover(self.sma2, self.sma1):
            self.sell()
MA1START = 2
MA1END = 100
MA1STEP = 1

MA2START = 3
MA2END = 200
MA2STEP = 1

def SMA(prices, n):
    return pd.Series(prices).rolling(n).mean()

def optimize_params(params):
    n1, n2 = params
    bt = Backtest(df, SmaCross, cash=10000, commission=0.004,
                 exclusive_orders=True, trade_on_close=True)
    stats = bt.run(n1=n1, n2=n2)
    return (n1, n2, stats['Sharpe Ratio'], stats['Return [%]'], stats['Max. Drawdown [%]'])

if __name__ == '__main__':
    # Load data (replace with your data)
    # df = pd.read_csv('your_data.csv', parse_dates=True, index_col=0)

    # Generate all parameter combinations
    param_combinations = [(n1, n2)
                         for n1 in range(MA1START, MA1END, MA1STEP)
                         for n2 in range(MA2START, MA2END, MA2STEP)
                         if n1 < n2]

    # Set up multiprocessing pool
    num_cores = 10  # Reserve 1 core for system
    print(f"Using {num_cores} CPU cores for parallel optimization...")

    # Perform parallel optimization using multiprocessing
    with mp.Pool(processes=num_cores) as pool:
        results = list(tqdm(pool.imap(optimize_params, param_combinations),
                          total=len(param_combinations),
                          desc="Optimization progress"))

    # Convert to DataFrame and sort
    results_df = pd.DataFrame(results,
                            columns=['n1', 'n2', 'Sharpe Ratio', 'Return [%]', 'Max. Drawdown [%]'])
    results_df = results_df.sort_values(by='Sharpe Ratio', ascending=False)

    # Output results
    print("\nTop 5 best parameter combinations:")
    print(results_df.head(5).to_string(index=False))

    # Best result details
    best = results_df.iloc[0]
    print(f"\nBest combination: SMA({best['n1']},{best['n2']})")
    print(f"Sharpe Ratio: {best['Sharpe Ratio']:.2f}")
    print(f"Annual Return: {best['Return [%]']:.2f}%")
    print(f"Max Drawdown: {best['Max. Drawdown [%]']:.2f}%")

Using 10 CPU cores for parallel optimization...


Optimization progress: 100%|██████████| 14553/14553 [14:23<00:00, 16.84it/s]



Top 5 best parameter combinations:
 n1  n2  Sharpe Ratio  Return [%]  Max. Drawdown [%]
 56  66      1.293526  235.213824         -26.886000
 55  69      1.262059  208.058252         -26.845405
 54  70      1.262059  208.058252         -26.845405
 54  69      1.262059  208.058252         -26.845405
 53  71      1.262059  208.058252         -26.845405

Best combination: SMA(56.0,66.0)
Sharpe Ratio: 1.29
Annual Return: 235.21%
Max Drawdown: -26.89%
